In [1]:
import pandas as pd 
import numpy as np 
import seaborn as sns
from matplotlib import pyplot as plt
import os 


In [2]:
df = pd.read_csv( "/Users/nishanrana/Ecommerce_Profitability_Analysis/data/Raw/Superstore.csv",
    encoding="latin1")

In [3]:
df.head

<bound method NDFrame.head of       Row ID        Order ID  Order Date   Ship Date       Ship Mode  \
0          1  CA-2016-152156   11/8/2016  11/11/2016    Second Class   
1          2  CA-2016-152156   11/8/2016  11/11/2016    Second Class   
2          3  CA-2016-138688   6/12/2016   6/16/2016    Second Class   
3          4  US-2015-108966  10/11/2015  10/18/2015  Standard Class   
4          5  US-2015-108966  10/11/2015  10/18/2015  Standard Class   
...      ...             ...         ...         ...             ...   
9989    9990  CA-2014-110422   1/21/2014   1/23/2014    Second Class   
9990    9991  CA-2017-121258   2/26/2017    3/3/2017  Standard Class   
9991    9992  CA-2017-121258   2/26/2017    3/3/2017  Standard Class   
9992    9993  CA-2017-121258   2/26/2017    3/3/2017  Standard Class   
9993    9994  CA-2017-119914    5/4/2017    5/9/2017    Second Class   

     Customer ID     Customer Name    Segment        Country             City  \
0       CG-12520       C

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

In [5]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 9994
Number of columns: 21


In [7]:
missing_values = df.isnull().sum()
missing_values[missing_values > 0].sort_values(ascending=False)


Series([], dtype: int64)

In [8]:
missing_percentage = (df.isnull().sum().sum() / df.size) * 100

print(f"Total missing value percentage: {missing_percentage:.2f}%")

Total missing value percentage: 0.00%


In [9]:
exact_duplicates = df.duplicated().sum()
print("Exact duplicate rows:", exact_duplicates)

Exact duplicate rows: 0


In [10]:
duplicates_without_id = df.drop(columns=["Row ID"]).duplicated().sum()
print("Duplicates excluding Row ID:", duplicates_without_id)

Duplicates excluding Row ID: 1


In [11]:
baseline = {
    "Rows": df.shape[0],
    "Columns": df.shape[1],
    "Missing Values": df.isnull().sum().sum(),
    "Exact Duplicates": df.duplicated().sum(),
    "Duplicates Excluding Row ID": df.drop(columns=["Row ID"]).duplicated().sum()
}

baseline_summary = pd.DataFrame(
    baseline.items(),
    columns=["Metric", "Value"]
)

baseline_summary

,Metric,Value
0,Rows,9994
1,Columns,21
2,Missing Values,0
3,Exact Duplicates,0
4,Duplicates Excluding Row ID,1


In [12]:
df.columns.tolist()

['Row ID',
 'Order ID',
 'Order Date',
 'Ship Date',
 'Ship Mode',
 'Customer ID',
 'Customer Name',
 'Segment',
 'Country',
 'City',
 'State',
 'Postal Code',
 'Region',
 'Product ID',
 'Category',
 'Sub-Category',
 'Product Name',
 'Sales',
 'Quantity',
 'Discount',
 'Profit']

In [13]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df.columns.tolist()

['row_id',
 'order_id',
 'order_date',
 'ship_date',
 'ship_mode',
 'customer_id',
 'customer_name',
 'segment',
 'country',
 'city',
 'state',
 'postal_code',
 'region',
 'product_id',
 'category',
 'sub-category',
 'product_name',
 'sales',
 'quantity',
 'discount',
 'profit']

In [14]:
print("Duplicate column names:", df.columns.duplicated().sum())

Duplicate column names: 0


In [17]:
text_columns = df.select_dtypes(include="object").columns.tolist()
text_columns

['order_id',
 'order_date',
 'ship_date',
 'ship_mode',
 'customer_id',
 'customer_name',
 'segment',
 'country',
 'city',
 'state',
 'region',
 'product_id',
 'category',
 'sub-category',
 'product_name']

In [18]:
nbsp_counts = df[text_columns].apply(
    lambda column: column.str.contains("\u00A0", na=False).sum()
)

nbsp_counts[nbsp_counts > 0]

product_name    225
dtype: int64

In [19]:
for column in text_columns:
    df[column] = (
        df[column]
        .str.replace("\u00A0", " ", regex=False)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

In [20]:
remaining_nbsp = df[text_columns].apply(
    lambda column: column.str.contains("\u00A0", na=False).sum()
).sum()

print("Remaining non-breaking spaces:", remaining_nbsp)

Remaining non-breaking spaces: 0


In [21]:
df.dtypes

row_id             int64
order_id          object
order_date        object
ship_date         object
ship_mode         object
customer_id       object
customer_name     object
segment           object
country           object
city              object
state             object
postal_code        int64
region            object
product_id        object
category          object
sub-category      object
product_name      object
sales            float64
quantity           int64
discount         float64
profit           float64
dtype: object

In [22]:
df["order_date"] = pd.to_datetime(
    df["order_date"],
    format="%m/%d/%Y",
    errors="coerce"
)

In [23]:
df["ship_date"] = pd.to_datetime(
    df["ship_date"],
    format="%m/%d/%Y",
    errors="coerce"
)

In [24]:
print("Order Date Type:", df["order_date"].dtype)
print("Ship Date Type:", df["ship_date"].dtype)

Order Date Type: datetime64[ns]
Ship Date Type: datetime64[ns]


In [25]:
print("Missing Order Dates:", df["order_date"].isnull().sum())
print("Missing Ship Dates:", df["ship_date"].isnull().sum())

Missing Order Dates: 0
Missing Ship Dates: 0


In [26]:
df["postal_code"] = df["postal_code"].astype("string")

In [27]:
print("Postal Code Type:", df["postal_code"].dtype)
df[["postal_code"]].head()

Postal Code Type: string


,postal_code
0,42420
1,42420
2,90036
3,33311
4,33311


In [28]:
df.dtypes


row_id                    int64
order_id                 object
order_date       datetime64[ns]
ship_date        datetime64[ns]
ship_mode                object
customer_id              object
customer_name            object
segment                  object
country                  object
city                     object
state                    object
postal_code      string[python]
region                   object
product_id               object
category                 object
sub-category             object
product_name             object
sales                   float64
quantity                  int64
discount                float64
profit                  float64
dtype: object

In [30]:
invalid_shipping_dates = df[
    df["ship_date"] < df["order_date"]
]

print("Invalid shipping date records:", len(invalid_shipping_dates))

Invalid shipping date records: 0


In [31]:
df["shipping_days"] = (
    df["ship_date"] - df["order_date"]
).dt.days

df[["order_date", "ship_date", "shipping_days"]].head()

,order_date,ship_date,shipping_days
0,2016-11-08,2016-11-11,3
1,2016-11-08,2016-11-11,3
2,2016-06-12,2016-06-16,4
3,2015-10-11,2015-10-18,7
4,2015-10-11,2015-10-18,7


In [32]:
df["shipping_days"].describe()

count    9994.000000
mean        3.958175
std         1.747567
min         0.000000
25%         3.000000
50%         4.000000
75%         5.000000
max         7.000000
Name: shipping_days, dtype: float64

In [33]:
comparison_columns = [
    col for col in df.columns
    if col not in ["row_id", "shipping_days"]
]

In [34]:
duplicate_mask = df.duplicated(
    subset=comparison_columns,
    keep=False
)

In [35]:
duplicate_records = df.loc[duplicate_mask]

print("Number of duplicate records:", len(duplicate_records))

duplicate_records.sort_values(
    by=["order_id", "product_id"]
)

Number of duplicate records: 2


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,region,product_id,category,sub-category,product_name,sales,quantity,discount,profit,shipping_days
3405,3406,US-2014-150119,2014-04-23,2014-04-27,Standard Class,LB-16795,Laurel Beltran,Home Office,United States,Columbus,...,East,FUR-CH-10002965,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.372,2,0.3,-12.0588,4
3406,3407,US-2014-150119,2014-04-23,2014-04-27,Standard Class,LB-16795,Laurel Beltran,Home Office,United States,Columbus,...,East,FUR-CH-10002965,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.372,2,0.3,-12.0588,4


In [36]:
repeated_pairs = df[
    df.duplicated(
        subset=["order_id", "product_id"],
        keep=False
    )
].sort_values(
    by=["order_id", "product_id"]
)

repeated_pairs[
    [
        "row_id",
        "order_id",
        "product_id",
        "product_name",
        "quantity",
        "sales",
        "discount",
        "profit"
    ]
]

,row_id,order_id,product_id,product_name,quantity,sales,discount,profit
6498,6499,CA-2015-103135,OFF-BI-10000069,"GBC Prepunched Paper, 19-Hole, for Binding Sys...",9,135.090,0.0,62.1414
6500,6501,CA-2015-103135,OFF-BI-10000069,"GBC Prepunched Paper, 19-Hole, for Binding Sys...",6,90.060,0.0,41.4276
350,351,CA-2016-129714,OFF-PA-10001970,Xerox 1881,2,24.560,0.0,11.5432
352,353,CA-2016-129714,OFF-PA-10001970,Xerox 1881,4,49.120,0.0,23.0864
1300,1301,CA-2016-137043,FUR-FU-10003664,"Electrix Architect's Clamp-On Swing Arm Lamp, ...",6,572.760,0.0,166.1004
1301,1302,CA-2016-137043,FUR-FU-10003664,"Electrix Architect's Clamp-On Swing Arm Lamp, ...",3,286.380,0.0,83.0502
9168,9169,CA-2016-140571,OFF-PA-10001954,Xerox 1964,14,319.760,0.0,147.0896
9169,9170,CA-2016-140571,OFF-PA-10001954,Xerox 1964,2,45.680,0.0,21.0128
7881,7882,CA-2017-118017,TEC-AC-10002006,Memorex Micro Travel Drive 16 GB,6,76.752,0.2,10.5534
7882,7883,CA-2017-118017,TEC-AC-10002006,Memorex Micro Travel Drive 16 GB,8,102.336,0.2,14.0712


In [37]:
product_name_counts = (
    df.groupby("product_id")["product_name"]
    .nunique()
)

conflicting_product_ids = product_name_counts[
    product_name_counts > 1
]

print(
    "Number of Product IDs with multiple names:",
    len(conflicting_product_ids)
)

Number of Product IDs with multiple names: 32


In [39]:
conflict_details = (
    df[
        df["product_id"].isin(conflicting_product_ids.index)
    ][
        ["product_id", "product_name", "category"]
    ]
    .drop_duplicates()
    .sort_values(["product_id", "product_name"])
)

conflict_details

,product_id,product_name,category
2115,FUR-BO-10002213,DMI Eclipse Executive Suite Bookcases,Furniture
2471,FUR-BO-10002213,"Sauder Forest Hills Library, Woodland Oak Finish",Furniture
128,FUR-CH-10001146,"Global Task Chair, Black",Furniture
66,FUR-CH-10001146,"Global Value Mid-Back Manager's Chair, Gray",Furniture
1459,FUR-FU-10001473,DAX Wood Document Frame,Furniture
...,...,...,...
1219,TEC-PH-10002200,Samsung Galaxy Note 2,Technology
1378,TEC-PH-10002310,Panasonic KX T7731-B Digital phone,Technology
2596,TEC-PH-10002310,Plantronics Calisto P620-M USB Wireless Speake...,Technology
2713,TEC-PH-10004531,AT&T CL2909,Technology
